<a href="https://colab.research.google.com/github/castral02/hatspots_TF/blob/main/hatspot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import esm
import torch
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

In [ ]:
#importing ESM embeddings
!pip install fair-esm torch
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
model.eval() # Set the model to evaluation mode
batch_converter = alphabet.get_batch_converter()

In [ ]:
import torch
import urllib.request

#grabbing scalers
import pickle
url = 'https://github.com/castral02/hatspots_TF/blob/main/trained_model/scaler_binder.pkl'
scalar_file = 'scaler_binder.pkl'
urllib.request.urlretrieve(url, scalar_file)
with open(scalar_file, 'rb') as f:
    scaler_binder = pickle.load(f)

url = 'https://github.com/castral02/hatspots_TF/blob/main/trained_model/scaler_domain.pkl'
domain_file = 'scaler_domain.pkl'
urllib.request.urlretrieve(url, domain_file)
with open(domain_file, 'rb') as f:
    scaler_domain = pickle.load(f)

In [3]:
#grabbing domains
protein_data = {
'Variant_ID': ['BRD7', 'P300', 'P300 KIX', 'P300 TAZ1', 'P300 TAZ2'],
'Domain Sequence': ['MGKKHKKHKSDKHLYEEYVEKPLKLVLKVGGNEVTELSTGSSGHDSSLFEDKNDHDKHKDRKRKKRKKGEKQIPGEEKGRKRRRVKEDKKKRDRDRVENEAEKDLQCHAPVRLDLPPEKPLTSSLAKQEEVEQTPLQEALNQLMRQLQRKDPSAFFSFPVTDFIAPGYSMIIKHPMDFSTMKEKIKNNDYQSIEELKDNFKLMCTNAMIYNKPETIYYKAAKKLLHSGMKILSQERIQSLKQSIDFMADLQKTRKQKDGTDTSQSGEDGGCWQREREDSGDAEAHAFKSPSKENKKKDKDMLEDKFKSNNLEREQEQLDRIVKESGGKLTRRLVNSQCEFERRKPDGTTTLGLLHPVDPIVGEPGYCPVRLGMTTGRLQSGVNTLQGFKEDKRNKVTPVLYLNYGPYSSYAPHYDSTFANISKDDSDLIYSTYGEDSDLPSDFSIHEFLATCQDYPYVMADSLLDVLTKGGHSRTLQEMEMSLPEDEGHTRTLDTAKEMEITEVEPPGRLDSSTQDRLIALKAVTNFGVPVEVFDSEEAEIFQKKLDETTRLLRELQEAQNERLSTRPPPNMICLLGPSYREMHLAEQVTNNLKELAQQVTPGDIVSTYGVRKAMGISIPSPVMENNFVDLTEDTEEPKKTDVAECGPGGS',
'MAENVVEPGPPSAKRPKLSSPALSASASDGTDFGSLFDLEHDLPDELINSTELGLTNGGDINQLQTSLGMVQDAASKHKQLSELLRSGSSPNLNMGVGGPGQVMASQAQQSSPGLGLINSMVKSPMTQAGLTSPNMGMGTSGPNQGPTQSTGMMNSPVNQPAMGMNTGMNAGMNPGMLAAGNGQGIMPNQVMNGSIGAGRGRQNMQYPNPGMGSAGNLLTEPLQQGSPQMGGQTGLRGPQPLKMGMMNNPNPYGSPYTQNPGQQIGASGLGLQIQTKTVLSNNLSPFAMDKKAVPGGGMPNMGQQPAPQVQQPGLVTPVAQGMGSGAHTADPEKRKLIQQQLVLLLHAHKCQRREQANGEVRQCNLPHCRTMKNVLNHMTHCQSGKSCQVAHCASSRQIISHWKNCTRHDCPVCLPLKNAGDKRNQQPILTGAPVGLGNPSSLGVGQQSAPNLSTVSQIDPSSIERAYAALGLPYQVNQMPTQPQVQAKNQQNQQPGQSPQGMRPMSNMSASPMGVNGGVGVQTPSLLSDSMLHSAINSQNPMMSENASVPSLGPMPTAAQPSTTGIRKQWHEDITQDLRNHLVHKLVQAIFPTPDPAALKDRRMENLVAYARKVEGDMYESANNRAEYYHLLAEKIYKIQKELEEKRRTRLQKQNMLPNAAGMVPVSMNPGPNMGQPQPGMTSNGPLPDPSMIRGSVPNQMMPRITPQSGLNQFGQMSMAQPPIVPRQTPPLQHHGQLAQPGALNPPMGYGPRMQQPSNQGQFLPQTQFPSQGMNVTNIPLAPSSGQAPVSQAQMSSSSCPVNSPIMPPGSQGSHIHCPQLPQPALHQNSPSPVPSRTPTPHHTPPSIGAQQPPATTIPAPVPTPPAMPPGPQSQALHPPPRQTPTPPTTQLPQQVQPSLPAAPSADQPQQQPRSQQSTAASVPTPTAPLLPPQPATPLSQPAVSIEGQVSNPPSTSSTEVNSQAIAEKQPSQEVKMEAKMEVDQPEPADTQPEDISESKVEDCKMESTETEERSTELKTEIKEEEDQPSTSATQSSPAPGQSKKKIFKPEELRQALMPTLEALYRQDPESLPFRQPVDPQLLGIPDYFDIVKSPMDLSTIKRKLDTGQYQEPWQYVDDIWLMFNNAWLYNRKTSRVYKYCSKLSEVFEQEIDPVMQSLGYCCGRKLEFSPQTLCCYGKQLCTIPRDATYYSYQNRYHFCEKCFNEIQGESVSLGDDPSQPQTTINKEQFSKRKNDTLDPELFVECTECGRKMHQICVLHHEIIWPAGFVCDGCLKKSARTRKENKFSAKRLPSTRLGTFLENRVNDFLRRQNHPESGEVTVRVVHASDKTVEVKPGMKARFVDSGEMAESFPYRTKALFAFEEIDGVDLCFFGMHVQEYGSDCPPPNQRRVYISYLDSVHFFRPKCLRTAVYHEILIGYLEYVKKLGYTTGHIWACPPSEGDDYIFHCHPPDQKIPKPKRLQEWYKKMLDKAVSERIVHDYKDIFKQATEDRLTSAKELPYFEGDFWPNVLEESIKELEQEEEERKREENTSNESTDVTKGDSKNAKKKNNKKTSKNKSSLSRGNKKKPGMPNVSNDLSQKLYATMEKHKEVFFVIRLIAGPAANSLPPIVDPDPLIPCDLMDGRDAFLTLARDKHLEFSSLRRAQWSTMCMLVELHTQSQDRFVYTCNECKHHVETRWHCTVCEDYDLCITCYNTKNHDHKMEKLGLGLDDESNNQQAAATQSPGDSRRLSIQRCIQSLVHACQCRNANCSLPSCQKMKRVVQHTKGCKRKTNGGCPICKQLIALCCYHAKHCQENKCPVPFCLNIKQKLRQQQLQHRLQQAQMLRRRMASMQRTGVVGQQQGLPSPTPATPTTPTGQQPTTPQTPQPTSQPQPTPPNSMPPYLPRTQAAGPVSQGKAAGQVTPPTPPQTAQPPLPGPPPAAVEMAMQIQRAAETQRQMAHVQIFQRPIQHQMPPMTPMAPMGMNPPPMTRGPSGHLEPGMGPTGMQQQPPWSQGGLPQPQQLQSGMPRPAMMSVAQHGQPLNMAPQPGLGQVGISPLKPGTVSQQALQNLLRTLRSPSSPLQQQQVLSILHANPQLLAAFIKQRAAKYANSNPQPIPGQPGMPQGQPGLQPPTMPGQQGVHSNPAMQNMNPMQAGVQRAGLPQQQPQQQLQPPMGGMSPQAQQMNMNHNTMPSQFRDILRRQQMMQQQQQQGAGPGIGPGMANHNQFQQPQGVGYPPQQQQRMQHHMQQMQQGNMGQIGQLPQALGAEAGASLQAYQQRLLQQQMGSPVQPNPMSPQQHMLPNQAQSPHLQGQQIPNSLSNQVRSPQPVPSPRPQSQPPHSSPSPRMQPQPSPHHVSPQTSSPHPGLVAAQANPMEQGHFASPDQNSMLSQLASNPGMANLHGASATDLGLSTDNSDLNSNLSQSTLDIH',
'GIRKQWHEDITQDLRNHLVHKLVQAIFPTPDPAALKDRRMENLVAYARKVEGDMYESANNRAEYYHLLAEKIYKIQKELE',
'DPEKRKLIQQQLVLLLHAHKCQRREQANGEVRQCNLPHCRTMKNVLNHMTHCQSGKSCQVAHCASSRQIISHWKNCTRHDCPVCLPL',
'GDSRRLSIQRCIQSLVHACQCRNANCSLPSCQKMKRVVQHTKGCKRKTNGGCPICKQLIALCCYHAKHCQENKCPVPFCLNI',
]
}

# Create a mapping dictionary from the protein_data
protein_domains = dict(zip(protein_data['Variant_ID'], protein_data['Domain Sequence']))

# User asking what domains
domains_input = input("What domains would you like to look into: ('BRD7', 'P300', 'P300 KIX', 'P300 TAZ1', 'P300 TAZ2'): ")
domains = [domain.strip() for domain in domains_input.split(',')]

# Get the sequences for the requested domains
domain_sequences = {}
for domain in domains:
    if domain in protein_domains:
        domain_sequences[domain] = protein_domains[domain]
    else:
        print(f"Warning: '{domain}' not found in protein_domains dictionary")

protein_df = pd.DataFrame(domain_sequences)

What domains would you like to look into: ('BRD7', 'P300', 'P300 KIX', 'P300 TAZ1', 'P300 TAZ2'): BRD7, P300
{'BRD7': 'MGKKHKKHKSDKHLYEEYVEKPLKLVLKVGGNEVTELSTGSSGHDSSLFEDKNDHDKHKDRKRKKRKKGEKQIPGEEKGRKRRRVKEDKKKRDRDRVENEAEKDLQCHAPVRLDLPPEKPLTSSLAKQEEVEQTPLQEALNQLMRQLQRKDPSAFFSFPVTDFIAPGYSMIIKHPMDFSTMKEKIKNNDYQSIEELKDNFKLMCTNAMIYNKPETIYYKAAKKLLHSGMKILSQERIQSLKQSIDFMADLQKTRKQKDGTDTSQSGEDGGCWQREREDSGDAEAHAFKSPSKENKKKDKDMLEDKFKSNNLEREQEQLDRIVKESGGKLTRRLVNSQCEFERRKPDGTTTLGLLHPVDPIVGEPGYCPVRLGMTTGRLQSGVNTLQGFKEDKRNKVTPVLYLNYGPYSSYAPHYDSTFANISKDDSDLIYSTYGEDSDLPSDFSIHEFLATCQDYPYVMADSLLDVLTKGGHSRTLQEMEMSLPEDEGHTRTLDTAKEMEITEVEPPGRLDSSTQDRLIALKAVTNFGVPVEVFDSEEAEIFQKKLDETTRLLRELQEAQNERLSTRPPPNMICLLGPSYREMHLAEQVTNNLKELAQQVTPGDIVSTYGVRKAMGISIPSPVMENNFVDLTEDTEEPKKTDVAECGPGGS', 'P300': 'MAENVVEPGPPSAKRPKLSSPALSASASDGTDFGSLFDLEHDLPDELINSTELGLTNGGDINQLQTSLGMVQDAASKHKQLSELLRSGSSPNLNMGVGGPGQVMASQAQQSSPGLGLINSMVKSPMTQAGLTSPNMGMGTSGPNQGPTQSTGMMNSPVNQPAMGMNTGMNAGMNPGMLAAGNGQGIMPNQVMNGSIGAGRGRQNMQYPNPGMGSAGNL

In [4]:
#you would need to upload the data you want to test
#make sure that the structure is Variant_ID and Domain Sequence. if not you would have to change the subsequent data
data_path = ''#change the pathway here
df = pd.read_csv(data_path)

In [ ]:
#grabbing esm embeddings:
#concatting df and protien data
df_new = pd.concat([df, protein_df])

import numpy as np
from tqdm import tqdm

data_list = list(zip(df_new['Variant_ID'], df_new['Domain Sequence']))

# Process in smaller batches
batch_size = 8  # Adjust based on your GPU/CPU memory
sequence_embeddings = []

for i in tqdm(range(0, len(data_list), batch_size)):
    batch_data = data_list[i:i+batch_size]
    batch_labels, batch_strs, batch_tokens = batch_converter(batch_data)
    batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)

    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[33], return_contacts=False)
        token_representations = results["representations"][33]

    # Process each sequence in the batch
    for j, (variant_id, seq) in enumerate(batch_data):
        seq_len = batch_lens[j] - 2
        full_repr = token_representations[j, 1:seq_len+1]
        pooled_repr = full_repr.mean(0)

        sequence_embeddings.append({
            'Variant_ID': variant_id,
            'pooled_embedding': pooled_repr.cpu().numpy()
        })
    #took out full embeddings due to kernal dying T.T

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
binder_protein = domains
variant_ids = df['Variant_ID']
df_final = []

for i in range(len(variant_ids)):
    variant_id = variant_ids.iloc[i]
    # Get variant embedding using iloc with integer index
    vi_embedding = sequence_embeddings.iloc[i]['pooled_embedding']

    for binder in binder_protein:
        # Get binder embedding - need to find the correct index for binder
        # This assumes binders are after variants in sequence_embeddings
        binder_idx = len(variant_ids) + binder_protein.index(binder)
        binder_embedding = sequence_embeddings.iloc[binder_idx]['pooled_embedding']

        # Get Kd value
        df_final.append({
            'Variant_ID': variant_id,
            'VI_pooled': vi_embedding,
            'Binder': binder,
            'Binder_pooled': binder_embedding        })

df_final = pd.DataFrame(df_final)

In [ ]:
# Convert list of embeddings to 2D arrays
VI_embeddings = np.stack(df_final['VI_pooled'].values)
Binder_embeddings = np.stack(df_final['Binder_pooled'].values)

# Apply the scalers
VI_scaled = scaler_binder.transform(VI_embeddings)
Binder_scaled = scaler_domain.transform(Binder_embeddings)

# Add scaled embeddings back to dataframe
df_final['VI_pooled_scaled'] = list(VI_scaled)
df_final['Binder_pooled_scaled'] = list(Binder_scaled)

print(df_final.head())

In [ ]:
class ep300_mlp(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout_rate=0.4):
        super(ep300_mlp, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.bn1 = nn.BatchNorm1d(hidden_size)  # Helps with generalization
        self.relu1 = nn.ReLU()
        self.dropout = nn.Dropout(dropout_rate)  # Regularization
        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout(x)
        x = self.output_layer(x)
        return x

In [ ]:
import torch
import pandas as pd
import numpy as np

try:
    # Load model
    combined_embeddings = np.concatenate([VI_embeddings, Binder_embeddings], axis=1)
    input_size = combined_embeddings.shape[1]
    model = ep300_mlp(input_size, hidden_size=8, output_size=1)
    #grabbing model from github repo
    url = 'https://github.com/castral02/hatspots_TF/blob/main/trained_model/trained_model.pth'
    filename = 'trained_model.pth'

    urllib.request.urlretrieve(url, filename)

    #loading pytorch model
    model = torch.load(filename, map_location=torch.device('cpu'))


    # Prepare data
    VI_embeddings = np.stack(df_final['VI_pooled_scaled'].values)
    Binder_embeddings = np.stack(df_final['Binder_pooled_scaled'].values)
    combined_embeddings = np.concatenate([VI_embeddings, Binder_embeddings], axis=1)

    print(f"Input shape: {combined_embeddings.shape}")

    # Make predictions
    X_tensor = torch.FloatTensor(combined_embeddings)
    with torch.no_grad():
        predictions = model(X_tensor)
        predictions = predictions.numpy().flatten()

    # Add to dataframe and save
    df_final['Predicted_Kd'] = predictions
    df_final.to_csv('predictions.csv', index=False)

    print(f"Successfully saved {len(predictions)} predictions to predictions.csv")

except Exception as e:
    print(f"Error: {e}")